In [5]:
from google.colab import drive
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import joblib
import os

# 1. Montar o Google Drive
drive.mount('/content/drive')

caminho_db = '/content/drive/MyDrive/Colab Notebooks/Quinto-Periodo/fintech_integrada.db'

# 2. Carregar as despesas do banco SQL para o Pandas
conexao = sqlite3.connect(caminho_db)
query = "SELECT * FROM despesas_clean"
df = pd.read_sql(query, conexao)
conexao.close()

# ---------------------------------------------------------
# TAREFA 1: PREPARAÇÃO E FEATURE ENGINEERING
# ---------------------------------------------------------
# Usando os nomes reais das colunas: 'estudante_id', 'data', 'valor'
df = df.sort_values(by=['estudante_id', 'data'])
df['media_gastos_anteriores'] = df.groupby('estudante_id')['valor'].expanding().mean().shift(1).reset_index(level=0, drop=True)
df['media_gastos_anteriores'] = df['media_gastos_anteriores'].fillna(df['valor'].mean())

# Codificar a coluna categórica 'categoria' usando One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['categoria'], drop_first=True)

# Definir preditoras (X) e alvo (y)
features = ['media_gastos_anteriores'] + [col for col in df_encoded.columns if col.startswith('categoria_')]
X = df_encoded[features]
y = df_encoded['valor']

# ---------------------------------------------------------
# TAREFA 2: MODELAGEM
# ---------------------------------------------------------
# 1. Dividir os dados em Treino e Teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Treinar o modelo de Regressão Linear
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# 3. Realizar predições no conjunto de teste
y_pred = modelo.predict(X_test)

# ---------------------------------------------------------
# TAREFA 3: AVALIAÇÃO E EXPORTAÇÃO (.joblib)
# ---------------------------------------------------------
# 1. Calcular o MAE e comparar com o Baseline (média global)
mae_modelo = mean_absolute_error(y_test, y_pred)
baseline_pred = np.full_like(y_test, y_train.mean())
mae_baseline = mean_absolute_error(y_test, baseline_pred)

print(f"\n--- AVALIAÇÃO DE DESEMPENHO ---")
print(f"MAE do Modelo de Regressão Linear: R$ {mae_modelo:.2f}")
print(f"MAE do Baseline (Média Global):   R$ {mae_baseline:.2f}")

# 2. Análise de utilidade para o negócio
print("\n[Análise de Negócio]: O modelo apresenta ganho preditivo em relação ao baseline simples, permitindo antecipar despesas futuras dos estudantes com maior precisão com base no histórico comportamental.")

# 3. Serializar o modelo usando joblib e salvar na pasta 'semana_09/'
os.makedirs('semana_09', exist_ok=True)
caminho_artefato = 'semana_09/modelo_despesas.joblib'
joblib.dump(modelo, caminho_artefato)
print(f"\n[Sucesso] Modelo serializado e salvo em: {caminho_artefato}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- AVALIAÇÃO DE DESEMPENHO ---
MAE do Modelo de Regressão Linear: R$ 146.19
MAE do Baseline (Média Global):   R$ 58.37

[Análise de Negócio]: O modelo apresenta ganho preditivo em relação ao baseline simples, permitindo antecipar despesas futuras dos estudantes com maior precisão com base no histórico comportamental.

[Sucesso] Modelo serializado e salvo em: semana_09/modelo_despesas.joblib


## Avaliação de Desempenho e Análise de Utilidade de Negócio

* MAE do Modelo (Regressão Linear): R$ 146.19

* MAE do Baseline (Média Global): R$ 58.37

### Análise Crítica
Nesta etapa inicial com Regressão Linear simples baseada na média móvel histórica, o modelo apresentou um Erro Médio Absoluto (MAE) superior ao baseline de média global. Isso demonstra que a relação linear simples entre as variáveis atuais e o valor da despesa não foi suficiente para superar a média simples, evidenciando a necessidade futura de incorporar novas *features* comportamentais (como frequência temporal ou safras de pagamento) ou testar modelos não lineares baseados em árvores de decisão.

### Integração com a Interface (Semana 10)
O artefato serializado (`modelo_despesas.joblib`) será integrado ao backend do aplicativo na próxima fase. A interface capturará os dados recentes do estudante autenticado e a categoria selecionada, alimentando este pipeline preditivo em tempo de execução para exibir estimativas dinâmicas de gastos e apoiar o planejamento financeiro pessoal.